<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
# <font color='blue'>Data Science Academy</font>
## <font color='blue'>Inteligência Aumentada com RAG, GraphRAG e Agentic RAG</font>
## <font color='blue'>Projeto 3</font>
### <font color='blue'>Aplicação Integrada com SLM (Small Language Model) e RAG Para Automação de Processos</font>

## Pacotes Python Usados no Projeto

In [1]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.4 MB/s eta 0:00:00


In [2]:
# Instala as dependências
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.6/306.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Imports
import torch
import gradio as gr
import pandas as pd
import transformers
import qdrant_client
import sentence_transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import warnings
warnings.filterwarnings('ignore')

In [4]:
set_seed(1234)

A variável de ambiente abaixo controla o paralelismo na tokenização, ou seja, se múltiplas threads devem ser usadas para processar textos em paralelo. Dependendo do ambiente e da carga de trabalho, ativar ou desativar esse paralelismo pode impactar o desempenho.

In [5]:
%env TOKENIZERS_PARALLELISM=True

env: TOKENIZERS_PARALLELISM=True


In [6]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy"

Author: Data Science Academy



## Carregando os Dados Para o Módulo de RAG

In [7]:
# Carrega os dados para o módulo de RAG
df_dsa = pd.read_csv('dataset.csv')

In [8]:
# Shape
df_dsa.shape

(16407, 3)

In [9]:
# Amostra
df_dsa.head()

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [10]:
# Vamos trabalhar com apenas 100 registros para deixar a app mais veloz.
# Fique à vontade para trabalhar com volumes de dados maiores.
ques_data = df_dsa['Question'].tolist()[:100]
answer_data = df_dsa['Answer'].tolist()[:100]

## Modelo de Embeddings

https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [11]:
# Define o modelo de embeddings
modelo_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Módulo de RAG com Banco de Dados Vetorial

https://qdrant.tech/

In [12]:
# Usa o modelo de embeddings para criar os vetores a partir dos dados
vetores = modelo_embedding.encode(ques_data)

In [13]:
# Cria o cliente para o banco de dados vetorial definido em memória
banco_vetorial = QdrantClient(":memory:")

In [14]:
# Cria a coleção no banco de dados vetorial
banco_vetorial.create_collection(collection_name = "doc_data",
                                 vectors_config = VectorParams(size = len(vetores[0]),
                                                               distance = Distance.COSINE))

True

Leia o ebook no Capítulo 7 com os detalhes sobre a distância cosseno.

In [15]:
# Upload dos dados para o banco de dados vetorial
banco_vetorial.upload_collection(collection_name = "doc_data",
                                 ids = [i for i in range(len(ques_data))],
                                 vectors = vetores)

## Módulo de Recuperação de Informação

In [16]:
# Define a função que recebe uma pergunta como entrada
def dsa_recupera_dados(question):

    # Carrega o modelo de transformação de sentença "all-mpnet-base-v2" da biblioteca SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

    # Codifica a pergunta em um vetor de embeddings usando o modelo
    ques_vector = model.encode(question)

    # Consulta o banco de dados vetorial com o vetor da pergunta, buscando documentos semelhantes
    result = banco_vetorial.query_points(collection_name = "doc_data", query = ques_vector)

    # Cria uma lista vazia para armazenar os IDs dos documentos mais semelhantes
    sim_ids = []

    # Itera sobre os resultados da consulta e adiciona os IDs dos documentos à lista
    for i in result.points:
        sim_ids.append(i.id)

    # Recupera o contexto do documento mais semelhante com base no primeiro ID da lista
    context = answer_data[sim_ids[0]]

    # Retorna o contexto do documento como resposta
    return context

## Módulo de Integração Entre SLM e RAG
<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [17]:
# Define a função 'dsa_llm_rag' que recebe uma pergunta e um contexto como entrada
def dsa_llm_rag(question, context):

    # Define o nome do modelo de linguagem a ser utilizado, no caso "TinyLlama-1.1B-Chat-v1.0"
    nome_llm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Carrega o tokenizador associado ao modelo usando o nome especificado
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)

    # Carrega o modelo de linguagem causal usando o nome especificado
    # Substitua "cuda" por "cpu" se executar localmente
    model = AutoModelForCausalLM.from_pretrained(nome_llm, device_map = "cuda")

    # Define o prompt da app, que inclui a pergunta e o contexto, instruindo o LLM
    # a responder com base no contexto
    chat = [{"role": "user", "content": f"this is question {question} asked by user you are a medical clinic assistant answer the question based on this context {context} in not more than 3-4 points"}]

    # Aplica o template do chat, tokeniza o prompt e converte para tensores PyTorch,
    # adicionando o prompt de geração de texto
    # Remova a parte final [.to("cuda")] se executar localmente
    token_inputs = tokenizer.apply_chat_template(chat,
                                                 tokenize = True,
                                                 return_tensors = "pt",
                                                 add_generation_prompt = True).to("cuda")

    # Gera a resposta do modelo a partir dos tokens de entrada, usando amostragem
    # e definindo limites como o número máximo de novos tokens e a temperatura (nível de criatividade)
    token_outputs = model.generate(input_ids = token_inputs,
                                   do_sample = True,
                                   max_new_tokens = 500,
                                   temperature = 1.5)

    # Extrai os novos tokens gerados que não fazem parte da entrada original
    new_tokens = token_outputs[0][token_inputs.shape[-1]:]

    # Decodifica os novos tokens em texto, ignorando tokens especiais
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens = True)

    # Retorna o texto decodificado como resposta
    return decoded_output

In [18]:
# Define a função 'dsa_gera_resultado' que recebe a entrada do usuário como parâmetro
def dsa_gera_resultado(user_input):

    # Verifica se o usuário forneceu uma entrada
    if user_input:

        # Recupera o contexto relevante chamando a função 'dsa_recupera_dados' com a entrada do usuário
        context = dsa_recupera_dados(user_input)

        # Gera e retorna a resposta chamando a função 'dsa_llm_rag' com a entrada do usuário e o contexto recuperado
        return dsa_llm_rag(user_input, context)

    # Caso a entrada do usuário esteja vazia, retorna uma mensagem pedindo para digitar uma pergunta
    else:
        return "Digite sua pergunta."

## Módulo da Aplicação Web Para Deploy

In [19]:
# Definindo a interface personalizada
webapp = gr.Interface(

    # Função que processa a entrada do usuário
    fn = dsa_gera_resultado,

    # Caixa de texto maior com um placeholder personalizado
    inputs = gr.Textbox(lines = 2, placeholder = "Digite sua pergunta aqui...", label = "Entrada"),

    # Caixa de texto de saída com um rótulo personalizado
    outputs = gr.Textbox(label = "Resposta"),

    # Título do aplicativo
    title = "DSA - Projeto 3",

    # Descrição personalizada
    description = "Este é uma aplicação de IA para automação de processo médico de triagem de pacientes.",

    # Exemplos pré-definidos
    examples = [["What is (are) Parasites - Schistosomiasis ?"]],

    # Tema mais compacto
    theme = "compact")

In [20]:
# Launch the Gradio interface
webapp.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89eed8d46520ae540c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
%reload_ext watermark
%watermark -a "Data Science Academy"

Author: Data Science Academy



In [22]:
#%watermark -v -m

In [23]:
#%watermark --iversions

# Fim